# PyAthena

Smoke test for [PyAthena](https://pypi.org/project/PyAthena/), the Python DB API 2.0 client for
Amazon Athena. Examples 1 to 5 run offline: a `boto3.Session` with dummy static credentials keeps
botocore from looking up real ones, and `botocore.stub.Stubber` stands in for the Amazon Athena
API. Example 5 uses PyAthena's SQLAlchemy dialect (`pip install "pyathena[sqlalchemy]"`) and is
skipped when SQLAlchemy is not installed.

Example 6 runs one real query. It needs network access and AWS credentials (enable both with
`--with network --with aws-credentials`) plus an S3 location for query results in
`AWS_ATHENA_S3_STAGING_DIR`, or a workgroup that has one in `AWS_ATHENA_WORK_GROUP`
(default `primary`). The query reads no table, but Amazon Athena may still bill a minimum charge.

## Example 1: DB API module interface and an offline connection

In [ ]:
import datetime
import os
from unittest import mock

import boto3
import pyathena
from pyathena.cursor import Cursor, DictCursor

print(f"PyAthena {pyathena.__version__}")

# PEP 249 module globals and exception hierarchy.
assert pyathena.apilevel == "2.0"
assert pyathena.paramstyle == "pyformat"
assert pyathena.threadsafety in (0, 1, 2, 3)
for name in ("Warning", "Error", "InterfaceError", "DatabaseError", "DataError",
             "OperationalError", "InternalError", "ProgrammingError", "NotSupportedError"):
    assert issubclass(getattr(pyathena, name), Exception), name
assert issubclass(pyathena.OperationalError, pyathena.DatabaseError)
assert issubclass(pyathena.DatabaseError, pyathena.Error)

# PEP 249 type objects compare equal to the type names Amazon Athena reports.
assert pyathena.STRING == "varchar" and pyathena.NUMBER == "bigint" and pyathena.DATETIME == "timestamp"
assert pyathena.STRING != "integer"
assert pyathena.Date(2024, 1, 31) == datetime.date(2024, 1, 31)

# Dummy static credentials: building the client then needs no network and no AWS account.
session = boto3.Session(aws_access_key_id="testing", aws_secret_access_key="testing", region_name="us-east-1")
conn = pyathena.connect(
    s3_staging_dir="s3://example-bucket/query-results",
    schema_name="sampledb",
    work_group="primary",
    region_name="us-east-1",
    session=session,
)
assert conn.s3_staging_dir.rstrip("/") == "s3://example-bucket/query-results"
assert conn.schema_name == "sampledb" and conn.work_group == "primary"
assert conn.client.meta.region_name == "us-east-1"

cursor = conn.cursor()
assert isinstance(cursor, Cursor)
assert cursor.description is None and cursor.rowcount == -1 and cursor.arraysize > 0
try:
    cursor.fetchone()
except pyathena.ProgrammingError as exc:
    print(f"fetch before execute: {type(exc).__name__}: {exc}")
else:
    raise AssertionError("fetchone() before execute() did not raise")
assert isinstance(conn.cursor(DictCursor), DictCursor)
with conn.cursor() as managed:
    assert isinstance(managed, Cursor)
conn.close()

# Queries need somewhere to put results: a staging directory or a workgroup.
with mock.patch.dict(os.environ):
    os.environ.pop("AWS_ATHENA_S3_STAGING_DIR", None)
    os.environ.pop("AWS_ATHENA_WORK_GROUP", None)
    try:
        pyathena.connect(region_name="us-east-1", session=session)
    except pyathena.ProgrammingError as exc:
        print(f"no result location: {exc}")
    else:
        raise AssertionError("connect() succeeded without s3_staging_dir or work_group")

## Example 2: Formatting query parameters

In [ ]:
import datetime
import decimal
import re

import pyathena
from pyathena.formatter import DefaultParameterFormatter

formatter = DefaultParameterFormatter()
sql = formatter.format(
    "SELECT * FROM events WHERE name = %(name)s AND n = %(n)s AND ok = %(ok)s "
    "AND id IN %(ids)s AND day = %(day)s AND ts = %(ts)s AND price = %(price)s AND note = %(note)s",
    {
        "name": "O'Reilly",
        "n": 42,
        "ok": True,
        "ids": [1, 2, 3],
        "day": datetime.date(2024, 1, 31),
        "ts": datetime.datetime(2024, 1, 31, 12, 34, 56, 789000),
        "price": decimal.Decimal("1.50"),
        "note": None,
    },
)
print(sql)
assert "name = 'O''Reilly'" in sql, "single quotes are not escaped by doubling"
assert "n = 42" in sql
assert re.search(r"ok = true\b", sql, re.IGNORECASE)
assert "id IN (1, 2, 3)" in sql
assert "day = DATE '2024-01-31'" in sql
assert "ts = TIMESTAMP '2024-01-31 12:34:56.789'" in sql
assert "price = DECIMAL '1.50'" in sql
assert re.search(r"note = null\b", sql, re.IGNORECASE)

# Without parameters the statement is passed through unchanged.
assert formatter.format("SELECT 1") == "SELECT 1"

# The pyformat paramstyle takes a mapping, not a sequence.
try:
    formatter.format("SELECT %(a)s", ["x"])
except pyathena.ProgrammingError as exc:
    print(f"sequence parameters rejected: {exc}")
else:
    raise AssertionError("a list of parameters was accepted")

## Example 3: Converting result values to Python types

In [ ]:
import datetime
import decimal

from pyathena.converter import DefaultTypeConverter

converter = DefaultTypeConverter()
cases = [
    ("boolean", "true", True),
    ("tinyint", "7", 7),
    ("integer", "42", 42),
    ("bigint", "9007199254740993", 9007199254740993),
    ("double", "1.5", 1.5),
    ("decimal", "1.50", decimal.Decimal("1.50")),
    ("varchar", "hello", "hello"),
    ("date", "2024-01-31", datetime.date(2024, 1, 31)),
    ("timestamp", "2024-01-31 12:34:56.789", datetime.datetime(2024, 1, 31, 12, 34, 56, 789000)),
    ("json", '{"a": [1, 2]}', {"a": [1, 2]}),
    ("varbinary", "61 62 63", b"abc"),
]
for type_name, raw, expected in cases:
    value = converter.convert(type_name, raw)
    assert value == expected and type(value) is type(expected), (type_name, raw, value)
    print(f"{type_name:10s} {raw!r:30s} -> {value!r}")

# NULL is None for every type.
for type_name, _, _ in cases:
    assert converter.convert(type_name, None) is None, type_name

# Complex types: how much is parsed into Python objects depends on the PyAthena release.
for type_name, raw in (("array", "[1, 2, 3]"), ("map", "{a=1, b=2}"), ("row", "{a=1, b=x}")):
    print(f"{type_name:10s} {raw!r:30s} -> {converter.convert(type_name, raw)!r}")

## Example 4: A query round trip against a stubbed Amazon Athena API

In [ ]:
import datetime

import boto3
import pyathena
from botocore.stub import Stubber
from pyathena.cursor import DictCursor

session = boto3.Session(aws_access_key_id="testing", aws_secret_access_key="testing", region_name="us-east-1")
conn = pyathena.connect(
    s3_staging_dir="s3://example-bucket/query-results/",
    schema_name="sampledb",
    region_name="us-east-1",
    session=session,
    poll_interval=0.01,
)

# Record the StartQueryExecution request PyAthena sends.
sent = []


def record_request(params, model, **kwargs):
    if model.name == "StartQueryExecution":
        sent.append(dict(params))


conn.client.meta.events.register("provide-client-params", record_request)

query_id = "0b1c2d3e-4f50-6172-8394-a5b6c7d8e9f0"
output_location = f"s3://example-bucket/query-results/{query_id}.csv"
submitted = datetime.datetime(2024, 1, 31, 12, 0, 0, tzinfo=datetime.timezone.utc)


def query_execution(state, reason=None):
    status = {"State": state, "SubmissionDateTime": submitted}
    if state == "SUCCEEDED":
        status["CompletionDateTime"] = submitted + datetime.timedelta(seconds=1)
    if reason:
        status["StateChangeReason"] = reason
    return {
        "QueryExecution": {
            "QueryExecutionId": query_id,
            "Query": "SELECT ...",
            "StatementType": "DML",
            "ResultConfiguration": {"OutputLocation": output_location},
            "QueryExecutionContext": {"Database": "sampledb"},
            "Status": status,
            "Statistics": {"EngineExecutionTimeInMillis": 120, "DataScannedInBytes": 0},
            "WorkGroup": "primary",
        }
    }


def column(name, type_name, precision, case_sensitive=False):
    # The ColumnInfo fields the service returns for each result column.
    return {
        "CatalogName": "hive", "SchemaName": "", "TableName": "", "Name": name, "Label": name,
        "Type": type_name, "Precision": precision, "Scale": 0, "Nullable": "UNKNOWN",
        "CaseSensitive": case_sensitive,
    }


columns = [column("id", "integer", 10), column("name", "varchar", 2147483647, True), column("joined", "date", 0)]
rows = [
    {"Data": [{"VarCharValue": "id"}, {"VarCharValue": "name"}, {"VarCharValue": "joined"}]},  # header row
    {"Data": [{"VarCharValue": "1"}, {"VarCharValue": "O'Reilly"}, {"VarCharValue": "2024-01-31"}]},
    {"Data": [{"VarCharValue": "2"}, {"VarCharValue": "Ada"}, {}]},  # a missing value is NULL
]
results = {"ResultSet": {"Rows": rows, "ResultSetMetadata": {"ColumnInfo": columns}}, "UpdateCount": 0}

with Stubber(conn.client) as stub:
    stub.add_response("start_query_execution", {"QueryExecutionId": query_id})
    stub.add_response("get_query_execution", query_execution("RUNNING"))
    stub.add_response("get_query_execution", query_execution("SUCCEEDED"))
    stub.add_response("get_query_results", results)

    cursor = conn.cursor(DictCursor)
    cursor.execute("SELECT id, name, joined FROM people WHERE name = %(name)s", {"name": "O'Reilly"})

    assert sent[-1]["QueryString"] == "SELECT id, name, joined FROM people WHERE name = 'O''Reilly'", sent
    assert sent[-1]["QueryExecutionContext"]["Database"] == "sampledb"
    assert sent[-1]["ResultConfiguration"]["OutputLocation"] == "s3://example-bucket/query-results/"
    assert cursor.query_id == query_id and cursor.state == "SUCCEEDED"
    assert cursor.output_location == output_location
    assert cursor.data_scanned_in_bytes == 0
    assert [desc[0] for desc in cursor.description] == ["id", "name", "joined"]
    assert cursor.description[0][1] == pyathena.NUMBER and cursor.description[2][1] == pyathena.DATE

    first = cursor.fetchone()
    rest = cursor.fetchall()
    assert first == {"id": 1, "name": "O'Reilly", "joined": datetime.date(2024, 1, 31)}, first
    assert rest == [{"id": 2, "name": "Ada", "joined": None}], rest
    assert cursor.fetchone() is None
    stub.assert_no_pending_responses()
print(f"sent: {sent[-1]['QueryString']}")
print(f"rows: {[first] + rest}")

# A query that fails on the service side raises OperationalError carrying the reason.
with Stubber(conn.client) as stub:
    stub.add_response("start_query_execution", {"QueryExecutionId": query_id})
    stub.add_response(
        "get_query_execution",
        query_execution("FAILED", "COLUMN_NOT_FOUND: line 1:8: Column 'nope' cannot be resolved"),
    )
    try:
        conn.cursor().execute("SELECT nope")
    except pyathena.OperationalError as exc:
        assert "COLUMN_NOT_FOUND" in str(exc), exc
        print(f"failed query: {type(exc).__name__}: {exc}")
    else:
        raise AssertionError("a FAILED query execution did not raise OperationalError")
    stub.assert_no_pending_responses()
conn.close()

## Example 5: The SQLAlchemy dialect

Needs `pip install "pyathena[sqlalchemy]"`; the cell only prints a note when SQLAlchemy is missing.

In [ ]:
try:
    import sqlalchemy as sa
except ImportError:
    sa = None
    print("SQLAlchemy is not installed: skipping the dialect checks")

if sa is not None:
    from sqlalchemy.dialects import registry
    from sqlalchemy.schema import CreateTable

    # PyAthena registers its dialects through entry points.
    dialect_cls = registry.load("awsathena.rest")
    assert dialect_cls.name == "awsathena", dialect_cls.name
    print(f"awsathena.rest -> {dialect_cls.__module__}.{dialect_cls.__name__}")

    # Creating an engine does not connect; the URL is turned into connect() arguments.
    url = sa.engine.make_url(
        "awsathena+rest://:@athena.us-east-1.amazonaws.com:443/sampledb"
        "?s3_staging_dir=s3%3A%2F%2Fexample-bucket%2Fquery-results%2F&work_group=primary"
    )
    engine = sa.create_engine(url)
    assert engine.dialect.name == "awsathena"
    _, connect_kwargs = engine.dialect.create_connect_args(url)
    assert connect_kwargs["region_name"] == "us-east-1", connect_kwargs
    assert connect_kwargs["schema_name"] == "sampledb"
    assert connect_kwargs["s3_staging_dir"] == "s3://example-bucket/query-results/"
    assert connect_kwargs["work_group"] == "primary"

    # Compile a Core query to the Amazon Athena SQL dialect.
    people = sa.Table(
        "people", sa.MetaData(),
        sa.Column("id", sa.Integer), sa.Column("name", sa.String), sa.Column("joined", sa.Date),
        schema="sampledb",
    )
    query = (
        sa.select(people.c.name, sa.func.count().label("n"))
        .where(people.c.id > 10)
        .group_by(people.c.name)
        .order_by(sa.desc("n"))
        .limit(5)
    )
    compiled = query.compile(dialect=engine.dialect)
    sql = str(compiled)
    print(sql)
    assert "FROM sampledb.people" in sql and "count(*) AS n" in sql and "GROUP BY" in sql
    assert "%(" in sql, "the dialect should use the pyformat paramstyle"
    assert sorted(compiled.params.values()) == [5, 10], compiled.params

    # DDL with table options: an external, partitioned Parquet table.
    events = sa.Table(
        "events", sa.MetaData(),
        sa.Column("id", sa.BigInteger),
        sa.Column("payload", sa.String),
        sa.Column("dt", sa.String, awsathena_partition=True),
        schema="sampledb",
        awsathena_location="s3://example-bucket/events/",
        awsathena_file_format="PARQUET",
    )
    ddl = str(CreateTable(events).compile(dialect=engine.dialect))
    print(ddl)
    assert "CREATE EXTERNAL TABLE sampledb.events" in ddl
    assert "PARTITIONED BY" in ddl and "STORED AS PARQUET" in ddl
    assert "LOCATION 's3://example-bucket/events/'" in ddl

## Example 6: A real query (network and AWS credentials)

Runs `SELECT` on constants, so no table or data is needed. Set `AWS_ATHENA_S3_STAGING_DIR` to an
S3 prefix you own, or `AWS_ATHENA_WORK_GROUP` to a workgroup with a result location.

In [ ]:
import datetime
import os

import boto3
import pyathena

region = boto3.Session().region_name or "us-east-1"
work_group = os.environ.get("AWS_ATHENA_WORK_GROUP", "primary")
# s3_staging_dir is left unset, so PyAthena reads AWS_ATHENA_S3_STAGING_DIR.
conn = pyathena.connect(region_name=region, work_group=work_group)
try:
    with conn.cursor() as cursor:
        cursor.execute("SELECT 1 AS one, %(word)s AS word, DATE '2024-01-31' AS day", {"word": "hello"})
        row = cursor.fetchone()
        assert row == (1, "hello", datetime.date(2024, 1, 31)), row
        assert cursor.state == "SUCCEEDED", cursor.state
        assert [desc[0] for desc in cursor.description] == ["one", "word", "day"]
        print(f"query {cursor.query_id} in {region}: {row}, {cursor.data_scanned_in_bytes} bytes scanned")
finally:
    conn.close()